In [3]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from kafka import KafkaConsumer

KAFKA_BOOTSTRAP_SERVERS = ["localhost:9092"]
INFERENCE_TOPIC = "ml_predictions_final"
GROUP_ID        = "inference_dashboard"
MAX_POINTS      = 200
SAVE_EVERY      = 1

timestamps       = []
predictions      = []
prob_aeration    = []
prob_no_aeration = []

consumer = KafkaConsumer(
    INFERENCE_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    auto_offset_reset="latest",
    enable_auto_commit=True,
    group_id=GROUP_ID,
)

from collections import defaultdict

# buffers por sensor
buffers = defaultdict(lambda: {
    "timestamps": [],
    "preds": [],
    "p0": [],
    "p1": [],
    "ts_pred1": [],  
    "conf_pred1": [],
    "ts_pred0": [],  
    "conf_pred0": [],
})

print(f"Subscribed to Kafka topic '{INFERENCE_TOPIC}'.\n")

msg_count = 0

try:
    for message in consumer:
        sensor_id = message.key if message.key else message.value.get("sensor_id", "unknown_sensor")
        sensor_data = message.value
        print(f"Received sensor={sensor_id}: {sensor_data}")

        ts = sensor_data["timestamp"]
        pred = int(sensor_data["prediction"])
        p0 = float(sensor_data["prob_class0"])
        p1 = float(sensor_data["prob_class1"])

        # buffer del sensor
        buf = buffers[sensor_id]

        # Update data storage
        buf["timestamps"].append(ts)
        buf["preds"].append(pred)
        buf["p0"].append(p0)
        buf["p1"].append(p1)

        # series filtradas por clase escogida
        if pred == 1:
            buf["ts_pred1"].append(ts)
            buf["conf_pred1"].append(p1)
        else:
            buf["ts_pred0"].append(ts)
            buf["conf_pred0"].append(p0)

        # Keep only the last 100 entries
        if len(buf["timestamps"]) > 100:
            min_ts = buf["timestamps"][-100]
            buf["timestamps"] = buf["timestamps"][-100:]
            buf["preds"] = buf["preds"][-100:]
            buf["p0"] = buf["p0"][-100:]
            buf["p1"] = buf["p1"][-100:]

            while buf["ts_pred1"] and buf["ts_pred1"][0] < min_ts:
                buf["ts_pred1"].pop(0); buf["conf_pred1"].pop(0)
            while buf["ts_pred0"] and buf["ts_pred0"][0] < min_ts:
                buf["ts_pred0"].pop(0); buf["conf_pred0"].pop(0)

        # Plot (2x2 con 3 paneles como la referencia)
        plt.figure(figsize=(10, 8))

        plt.subplot(2, 2, 1)
        plt.plot(buf["timestamps"], buf["preds"])
        plt.title("Aeration")
        plt.ylabel("Aeration produced")
        plt.ylim(-0.05, 1.05)

        plt.subplot(2, 2, 2)
        plt.plot(buf["ts_pred1"], buf["conf_pred1"], color="green")
        plt.title("Aeration confidence")
        plt.ylabel("Confidence")
        plt.ylim(0.0, 1.05)

        plt.subplot(2, 2, 3)
        plt.plot(buf["ts_pred0"], buf["conf_pred0"], color="orange")
        plt.title("Non aeration confidence")
        plt.ylabel("Confidence")
        plt.ylim(0.0, 1.05)

        plt.subplot(2, 2, 4)
        plt.axis("off")

        plt.tight_layout()
        plt.savefig(f"Images/Parte3/aeration_inference_{sensor_id.decode('utf-8')}.png")
        plt.close()


except KeyboardInterrupt:
    print("Stopped consuming messages.")
    consumer.close()

Subscribed to Kafka topic 'ml_predictions_final'.

Received sensor=b'sensor_002': {'sensor_id': 'sensor_002', 'timestamp': 1772385471, 'prediction': 0, 'prob_class0': 1.0, 'prob_class1': 0.0, 'water_temperature': 29.33, 'ph_level': 8.06, 'turbidity': 48.54, 'dissolved_oxygen': 11.71}
Received sensor=b'sensor_007': {'sensor_id': 'sensor_007', 'timestamp': 1772385471, 'prediction': 0, 'prob_class0': 1.0, 'prob_class1': 0.0, 'water_temperature': 29.76, 'ph_level': 8.56, 'turbidity': 7.29, 'dissolved_oxygen': 8.29}
Received sensor=b'sensor_009': {'sensor_id': 'sensor_009', 'timestamp': 1772385471, 'prediction': 1, 'prob_class0': 0.0, 'prob_class1': 1.0, 'water_temperature': 29.88, 'ph_level': 7.94, 'turbidity': 37.51, 'dissolved_oxygen': 5.59}
Received sensor=b'sensor_003': {'sensor_id': 'sensor_003', 'timestamp': 1772385471, 'prediction': 1, 'prob_class0': 0.0, 'prob_class1': 1.0, 'water_temperature': 29.13, 'ph_level': 8.19, 'turbidity': 47.51, 'dissolved_oxygen': 5.36}
Received sensor=b

AttributeError: eventson